# Generate the possession submission

Run all cells in order. Set `MTAH_DATA_DIR` to the folder containing the provided tactical ATD TXT/XML files. The frozen pipeline needs no training, internet, broadcast video, audio, events or possession labels. Outputs and optional caches remain under ignored `local_review/`.

CPU is the reference execution device. Every weight and input file is checked; the final CSV must match the frozen output checksum.

In [ ]:
from pathlib import Path
import os
import sys
import json
import hashlib
import platform
import time
from datetime import datetime, timezone

import nbformat
import numpy as np
import pandas as pd
import torch

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'sol/final/config.json').is_file())
SOLUTION = ROOT / 'sol/final'
CONFIG = json.loads((SOLUTION / 'config.json').read_text())
DATA = Path(os.environ.get('MTAH_DATA_DIR', ROOT / 'data/tactical')).expanduser().resolve()
OUTPUT = ROOT / 'local_review/submission'
CACHE = ROOT / 'local_review/cache'
OUTPUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
USE_CACHE = os.environ.get('MTAH_USE_CACHE', '1') == '1'
TEAM_NAME = os.environ.get('MTAH_TEAM_NAME', CONFIG['team_name']).strip()
if not TEAM_NAME or Path(TEAM_NAME).name != TEAM_NAME or TEAM_NAME in ('.', '..'):
    raise ValueError('MTAH_TEAM_NAME must be a nonempty file name without directory separators.')
if CONFIG['inference']['device'] != 'cpu':
    raise ValueError('This frozen reproduction path is verified on CPU.')
torch.set_num_threads(CONFIG['inference']['threads'])

# Import only this repository's definition notebooks, never an executed research notebook.
for name in ('features.ipynb', 'models.ipynb'):
    document = nbformat.read(SOLUTION / name, as_version=4)
    for cell in document.cells:
        if cell.cell_type == 'code':
            exec(compile(cell.source, name, 'exec'), globals())

weights_manifest = json.loads((SOLUTION / 'weights/manifest.json').read_text())
for name, record in weights_manifest.items():
    if file_sha256(SOLUTION / 'weights' / name) != record['sha256']:
        raise ValueError(f'Weight or preprocessing checksum mismatch: {name}')
raw = DATA / CONFIG['match']['tracking_file']
metadata = DATA / CONFIG['match']['metadata_file']
if not raw.is_file() or not metadata.is_file():
    raise FileNotFoundError(f'Set MTAH_DATA_DIR to the directory containing the tactical ATD TXT and XML: {DATA}')
for path, expected in [(raw, CONFIG['match']['tracking_sha256']),
                       (metadata, CONFIG['match']['metadata_sha256'])]:
    if file_sha256(path) != expected:
        raise ValueError(f'Unexpected input file: {path.name}')

def notebook_code_hash(path):
    cells = nbformat.read(path, as_version=4).cells
    source = '\n'.join(cell.source for cell in cells if cell.cell_type == 'code')
    return hashlib.sha256(source.encode()).hexdigest()

signature = {
    'config': CONFIG,
    'components': {name: notebook_code_hash(SOLUTION / name)
                   for name in ('features.ipynb', 'models.ipynb', 'inference.ipynb')},
    'weights': {name: entry['sha256'] for name, entry in weights_manifest.items()},
    'versions': {'python': platform.python_version(), 'numpy': np.__version__,
                 'pandas': pd.__version__, 'torch': str(torch.__version__)},
}
fingerprint = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()
cache_directory = CACHE / fingerprint[:16]
cache_directory.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()
print('Frozen model:', CONFIG['reference_model'])
print('Device: CPU; cache:', 'enabled' if USE_CACHE else 'disabled')


def cached_stage(name, compute):
    """Optional, fingerprinted acceleration. Every stage can rebuild from raw inputs."""
    path = cache_directory / (name + '.npz')
    sidecar = cache_directory / (name + '.json')
    if USE_CACHE and path.is_file() and sidecar.is_file():
        saved = json.loads(sidecar.read_text())
        if saved['fingerprint'] != fingerprint or saved['sha256'] != file_sha256(path):
            raise ValueError(f'Incompatible or corrupted cache: {name}. Remove this cache directory to rebuild.')
        with np.load(path, allow_pickle=False) as data:
            result = {key: data[key].copy() for key in data.files}
        print('Reused:', name)
        return result
    print('Computing:', name, flush=True)
    result = compute()
    np.savez_compressed(path, **result)
    sidecar.write_text(json.dumps({'fingerprint': fingerprint, 'sha256': file_sha256(path)}, indent=2))
    return result


def load_weights(model, filename):
    state = torch.load(SOLUTION / 'weights' / filename, map_location='cpu', weights_only=True)
    model.load_state_dict(state, strict=True)
    return model.eval()


## 1. Player-only D/LIVE inference
Rebuild the canonical timeline and player features, then average the two frozen D models and apply the learned duration decoder.

In [ ]:
data = cached_stage('geometry', lambda: read_tracking(raw, metadata, CONFIG['match']))
features = cached_stage('player_features', lambda: {'features': player_features(data)})['features']
with np.load(SOLUTION / 'weights/dead_scaler.npz') as scaler:
    dead_mean, dead_scale = scaler['mean'], scaler['scale']
normalized = (features - dead_mean) / dead_scale


def dead_probabilities():
    gru = load_weights(PlayerBiGRU(CONFIG['dead']), 'dead_bigru.pt')
    tcn = load_weights(PlayerTCN(CONFIG['dead']), 'dead_tcn.pt')
    batch_size = CONFIG['inference']['dead_batch_size']
    p_gru = predict_centered(gru, normalized, data, CONFIG['dead'], batch_size)[:, 1]
    p_tcn = predict_centered(tcn, normalized, data, CONFIG['dead'], batch_size)[:, 1]
    # Keep the original averaging order and float32 precision.
    p_dead = np.mean(np.stack([p_gru, p_tcn]), axis=0).astype(np.float32)
    return {'bigru': p_gru, 'tcn': p_tcn, 'probability': p_dead}

p_dead = cached_stage('dead_probabilities', dead_probabilities)['probability']
duration = json.loads((SOLUTION / 'weights/duration.json').read_text())
dead = duration_decode(p_dead, data, duration['weight'],
                       {'stride': duration['stride']}, np.asarray(duration['hazards'])).astype(bool)


## 2. Team possession and final labels
Infer H/A from observed player and ball geometry. Decode team possession separately within each predicted LIVE interval. Save all 199319 frame labels and verify exact reproduction.

In [ ]:
def encoder_outputs():
    encoder = load_weights(PossessionEncoder(), 'possession_encoder.pt')
    embedding, probability = predict_encoder(encoder, data, CONFIG['inference']['encoder_batch_size'])
    return {'embedding': embedding, 'probability': probability}

encoded = cached_stage('possession_embeddings', encoder_outputs)
with np.load(SOLUTION / 'weights/possession_scaler.npz') as scaler:
    ha_mean, ha_scale = scaler['mean'], scaler['scale']
ha_features = np.concatenate([(encoded['embedding'] - ha_mean) / ha_scale,
                              encoded['probability']], axis=1).astype(np.float32)


def possession_probabilities():
    model = load_weights(PossessionBiGRU(CONFIG['possession']), 'possession_bigru.pt')
    probability = predict_centered(model, ha_features, data, CONFIG['possession'],
                                   CONFIG['inference']['possession_batch_size'], possession=True)
    return {'probability': probability}

p_ha = cached_stage('possession_probabilities', possession_probabilities)['probability']
ha = decode_ha(p_ha, data, CONFIG['possession']['switch_penalty'], live=~dead)
mapping = data['mapping_25hz']
valid = mapping >= 0
labels = np.full(CONFIG['match']['frames'], 'X', dtype='<U1')
labels[valid] = np.array(['H', 'A'])[ha[mapping[valid]]]
# State the indexing directly: all in-half rows get H/A, then decoded D overrides them.
dead_rows = np.flatnonzero(valid)[dead[mapping[valid]]]
labels[dead_rows] = 'D'
submission = pd.DataFrame({'frame': np.arange(len(labels)), 'label': labels})
assert np.array_equal(submission.frame, np.arange(CONFIG['match']['frames']))
assert submission.label.isin(['H', 'A', 'D', 'X']).all() and (labels[~valid] == 'X').all()

# Use the unmodified official validation function, with no reference labels.
sys.path.insert(0, str(ROOT / 'vendor'))
import possession_metrics as metrics
error = metrics.validate_submission(submission, CONFIG['match']['frames'])
if error:
    raise ValueError(error)
output_path = OUTPUT / (TEAM_NAME + '.csv')
submission.to_csv(output_path, index=False, lineterminator='\n')
output_sha = file_sha256(output_path)
expected = CONFIG['expected_submission_sha256']
if output_sha != expected:
    raise RuntimeError('The generated CSV differs from the frozen reference output. '
                       'Do not submit it without investigating the environment or code changes.')
manifest = {'model': CONFIG['reference_model'], 'fingerprint': fingerprint,
            'created_utc': datetime.now(timezone.utc).isoformat(),
            'rows': len(labels), 'submission_sha256': output_sha,
            'exact_frozen_reproduction': True, 'reference_labels_read': False,
            'seconds': time.perf_counter() - started, 'signature': signature}
(OUTPUT / 'run_manifest.json').write_text(json.dumps(manifest, indent=2))
print('Validated:', output_path)
print('Exact frozen reproduction:', output_sha)
print(f'Total time: {manifest["seconds"]:.1f} seconds')
